# 1 - Baselines simples — média, mediana e propagação

## Objetivo

Implementar e avaliar os **baselines triviais** de imputação. Eles são os pisos absolutos da Etapa 4: se a GAIN não superar uma média global, há problema sério de implementação ou de protocolo.

> **Pergunta que responde:** quão bem se sai uma imputação "estúpida"? Qual o RMSE mínimo que precisamos superar?

## Posição na Etapa 3

Notebook **1 de 2** — consome os splits da Etapa 2 (`Data/GoldData/Splited`), a máscara real de test (`Data/GoldData/Masked`) e os artefatos de escala (`Data/ProcessedData`). Entrega 4 arquivos de predições + `tabelas/baselines_simples.csv`.

## Os 4 baselines

| # | Método | O que assume sobre o dado | Quando faria sentido na prática |
|---|--------|---------------------------|--------------------------------|
| 1 | Média global | Cada variável flutua em torno de um nível único | Variáveis estacionárias e homogêneas entre estações |
| 2 | Média por estação | O nível é específico de cada ponto de coleta | Heterogeneidade espacial forte (esperado aqui — ver EDA §5) |
| 3 | Mediana global | Como (1), mas robusta a caudas longas | Distribuições assimétricas (menos relevante após Box-Cox/Yeo-Johnson) |
| 4 | Forward fill por estação | O valor persiste entre coletas consecutivas | Autocorrelação temporal alta relativa ao intervalo amostral (~bimestral) |

## Protocolo de avaliação (comum à Etapa 3)

1. Sobre o `test`, gerar máscara artificial `B` com `generate_artificial_mask` (mesma função do treino da GAIN) — 3 sementes: `[42, 7, 2026]`.
2. Esconder as células `(M=1, B=0)`: o método só vê `X_obs = X` onde `M·B = 1`.
3. Imputar a matriz completa no espaço normalizado.
4. Desnormalizar verdade e predição (`MinMaxScaler⁻¹` → `Box-Cox/Yeo-Johnson⁻¹`) para que o erro esteja na **escala física original**.
5. RMSE e MAE **apenas** nas posições escondidas `(M=1, B=0)`; média ± desvio entre as 3 sementes.

A lógica vive em `baseline_utils.py` (mesmo padrão do `mask_utils.py` da Etapa 2: módulo importável em `Code/`), para que `02_baselines_ml.ipynb` use **exatamente o mesmo protocolo** — pré-condição do critério de aceite ("números comparáveis").

## Adaptações em relação ao plano (`Pipeline/03_Baselines/01_baselines_simples.md`)

- **Caminhos reais**: o plano cita `Data/Pipeline/02_processed/`; os artefatos da Etapa 2 vivem em `Data/GoldData/` e `Data/ProcessedData/`. Predições vão para `Data/BaselineResults/`.
- **11 variáveis, não 13**: a base foi reduzida em 2026-06-29 (removidas Cianobactérias e Microcistinas). O critério de aceite passa de 156 para 4 × 11 × 3 = 132 linhas — **menos** as de Coliformes Termotolerantes, que tem **0 observações no test** (gap consciente registrado em `split_info.json`) e portanto não gera métrica. Esperado: 4 × 10 × 3 = **120 linhas**.
- **`miss_rate = 0,20` uniforme, não `MISS_RATES_PADRAO` por variável**: as taxas por variável da Etapa 2 protegem o *sinal de treino* da GAIN nas variáveis raras. Aqui mascaramos o test **apenas para avaliar** — nada é treinado sobre ele — e a taxa uniforme maximiza as células avaliáveis justamente nas variáveis de baixa cobertura (SST tem só 16 observações no test). Reportamos `n_avaliado` para sinalizar métricas ruidosas. **A Etapa 5 deve reutilizar este mesmo protocolo (mesmas sementes, mesma taxa) ao avaliar a GAIN.**

## Setup

Imports + caminhos relativos a `Code/4 - Baseline/`. `mask_utils` (Etapa 2) entra via `sys.path` — decisão registrada em `Code/3 - Preprocessing/resumo.md`.

In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

sys.path.append("../3 - Preprocessing")
from mask_utils import generate_artificial_mask

import baseline_utils as bu

IN_SPLIT_DIR    = Path("../../Data/GoldData/Splited")
IN_MASK_DIR     = Path("../../Data/GoldData/Masked")
IN_COLUMNS_JSON = Path("../../Data/ProcessedData/encoded_columns.json")
IN_SCALERS      = Path("../../Data/ProcessedData/scalers.pkl")
IN_TRANSFORM    = Path("../../Data/ProcessedData/transform_params.json")

OUT_PRED_DIR = Path("../../Data/BaselineResults")
OUT_TAB_DIR  = Path("./tabelas")
OUT_PRED_DIR.mkdir(parents=True, exist_ok=True)
OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)

SEEDS     = [42, 7, 2026]
MISS_RATE = 0.20

## Carregamento

- `train` fornece as **estatísticas** (média, mediana, média por estação) — nunca o test, para não vazar informação do próprio conjunto avaliado.
- `test` é onde mascaramos e medimos.
- Sanidade: a máscara real persistida deve bater com o padrão de NaN do test (consistência entre artefatos da Etapa 2).

In [2]:
with open(IN_COLUMNS_JSON, encoding="utf-8") as f:
    encoded_columns = json.load(f)
VARS = encoded_columns["numericas"]

df_train = pd.read_parquet(IN_SPLIT_DIR / "train.parquet")
df_test  = pd.read_parquet(IN_SPLIT_DIR / "test.parquet")
mask_real_test = pd.read_parquet(IN_MASK_DIR / "mask_real_test.parquet")

# Pickle interno do projeto (gerado por 04_normalizacao.ipynb) — fonte confiável
scalers = joblib.load(IN_SCALERS)
with open(IN_TRANSFORM, encoding="utf-8") as f:
    transform_params = json.load(f)

# Sanidade: máscara persistida == padrão de NaN do test, colunas na mesma ordem
assert list(mask_real_test.columns) == VARS, "Ordem de colunas da máscara difere de VARS."
assert (mask_real_test.values == (~df_test[VARS].isna()).astype("int8").values).all(), \
    "mask_real_test não bate com o padrão de NaN de test.parquet."

print(f"train: {df_train.shape} | test: {df_test.shape} | variáveis-alvo: {len(VARS)}")

cobertura = mask_real_test.sum().rename("obs_test").to_frame()
cobertura["céls. escondidas esperadas/seed (≈20%)"] = (cobertura["obs_test"] * MISS_RATE).round(1)
cobertura

train: (515, 36) | test: (90, 36) | variáveis-alvo: 11


,obs_test,céls. escondidas esperadas/seed (≈20%)
DBO,78,15.6000
OD,74,14.8000
Nitrato,73,14.6000
Nitrogênio Amoniacal Total,77,15.4000
Fósforo Total,60,12.0000
Condutividade,72,14.4000
pH,79,15.8000
Turbidez,70,14.0000
Temperatura da Água,76,15.2000
Sólidos Suspensos Totais,16,3.2000


### Leitura da cobertura

Coliformes Termotolerantes tem **0 observações no test** (o INEA descontinuou a medição nas coletas de 2024–2025) — nenhum baseline gera métrica para ela. Sólidos Suspensos Totais tem ~16 observações → ~3 células avaliáveis por seed: a métrica existe, mas é **ruidosa** (por isso `n_avaliado` acompanha cada linha do CSV).

## Volta à escala original — sanidade

Antes de medir qualquer erro, verificamos que a cadeia de inversão (`clip [-1,1]` → `MinMaxScaler⁻¹` → `Box-Cox/Yeo-Johnson⁻¹`, implementada em `baseline_utils.desnormalizar`) devolve valores fisicamente plausíveis: pH em ~6–9, OD em mg/L de água superficial, temperatura em °C tropicais.

In [3]:
X_true_orig = bu.desnormalizar(df_test[VARS], VARS, scalers, transform_params)
X_true_orig.describe().T[["count", "mean", "min", "max"]]

,count,mean,min,max
DBO,78.0000,14.3185,0.0000,77.0000
OD,74.0000,6.3149,2.0000,20.0000
Nitrato,73.0000,0.1126,0.0100,0.5200
Nitrogênio Amoniacal Total,77.0000,2.7668,0.0000,8.7100
Fósforo Total,60.0000,70.7882,0.2300,1200.0000
Condutividade,72.0000,21572.3611,21.0000,52850.0000
pH,79.0000,7.9614,6.8300,9.5700
Turbidez,70.0000,58.4217,1.4500,887.0000
Temperatura da Água,76.0000,24.6053,19.8000,30.5000
Sólidos Suspensos Totais,16.0000,53.8125,16.0000,97.0000


## Baseline 1 — Média global

Substitui cada célula faltante pela **média da variável no train**. É o piso mais baixo que existe: ignora estação, ignora tempo, ignora as demais variáveis. Faria sentido na prática apenas para variáveis estacionárias e espacialmente homogêneas — o que a EDA já mostrou **não** ser o caso da lagoa (heterogeneidade forte entre estações, §5 do resumo da EDA).

No espaço normalizado a média do train não é 0 por construção (o scaler foi ajustado no dataset completo, decisão nº 1 da Etapa 2), por isso calculamos a média explicitamente em vez de preencher com zero.

In [4]:
media_train = df_train[VARS].mean()

def predict_media_global(X_obs, seed=None):
    return X_obs.fillna(media_train)

met_media_global, pred = bu.avaliar_baseline(
    "media_global", predict_media_global, df_test, mask_real_test,
    VARS, scalers, transform_params, SEEDS, MISS_RATE, generate_artificial_mask)

pred.to_parquet(OUT_PRED_DIR / "predictions_media_global.parquet", index=False)
met_media_global.groupby("variavel", sort=False)[["RMSE", "MAE", "n_avaliado"]].mean().round(4)

,RMSE,MAE,n_avaliado
variavel,,,
DBO,10.7571,6.8507,14.3333
OD,4.0230,3.4410,12.3333
Nitrato,0.1642,0.0913,14.3333
Nitrogênio Amoniacal Total,2.7745,2.4951,13.3333
Fósforo Total,108.9211,34.1428,10.6667
Condutividade,15836.5424,12926.8162,13.6667
pH,0.6633,0.5295,17.6667
Turbidez,153.4077,56.3346,18.0000
Temperatura da Água,3.4617,3.0640,15.6667


## Baseline 2 — Média por estação

Média da variável **condicional à estação de coleta** (recuperada do one-hot `est_*`), calculada no train. Se este baseline ganhar da média global com folga, é a confirmação quantitativa da heterogeneidade espacial vista na EDA — e um recado para a GAIN: o one-hot de estação carrega sinal.

Fallback: combinações variável × estação sem nenhuma observação no train caem para a média global.

In [5]:
estacao_train = bu.coluna_estacao(df_train)
estacao_test  = bu.coluna_estacao(df_test)
media_por_estacao = df_train[VARS].groupby(estacao_train.to_numpy()).mean()

def predict_media_estacao(X_obs, seed=None):
    base = media_por_estacao.reindex(estacao_test.to_numpy())
    base.index = X_obs.index
    return X_obs.fillna(base).fillna(media_train)

met_media_estacao, pred = bu.avaliar_baseline(
    "media_estacao", predict_media_estacao, df_test, mask_real_test,
    VARS, scalers, transform_params, SEEDS, MISS_RATE, generate_artificial_mask)

pred.to_parquet(OUT_PRED_DIR / "predictions_media_estacao.parquet", index=False)
met_media_estacao.groupby("variavel", sort=False)[["RMSE", "MAE", "n_avaliado"]].mean().round(4)

,RMSE,MAE,n_avaliado
variavel,,,
DBO,10.5858,7.0086,14.3333
OD,4.0635,3.3872,12.3333
Nitrato,0.1640,0.0921,14.3333
Nitrogênio Amoniacal Total,3.0489,2.7615,13.3333
Fósforo Total,108.8579,34.0318,10.6667
Condutividade,10200.5089,7371.1231,13.6667
pH,0.6485,0.5177,17.6667
Turbidez,145.0198,49.1845,18.0000
Temperatura da Água,3.5092,3.0975,15.6667


## Baseline 3 — Mediana global

Como o baseline 1, mas com a **mediana** do train — robusta a caudas longas. Nota: as transformações Box-Cox/Yeo-Johnson da Etapa 2 já simetrizaram boa parte das distribuições no espaço normalizado, então a diferença para a média tende a ser pequena; a exceção esperada é Fósforo Total, cuja cauda sobreviveu à transformação (kurtosis residual 8,9 — alerta duplo da Etapa 2).

In [6]:
mediana_train = df_train[VARS].median()

def predict_mediana_global(X_obs, seed=None):
    return X_obs.fillna(mediana_train)

met_mediana_global, pred = bu.avaliar_baseline(
    "mediana_global", predict_mediana_global, df_test, mask_real_test,
    VARS, scalers, transform_params, SEEDS, MISS_RATE, generate_artificial_mask)

pred.to_parquet(OUT_PRED_DIR / "predictions_mediana_global.parquet", index=False)
met_mediana_global.groupby("variavel", sort=False)[["RMSE", "MAE", "n_avaliado"]].mean().round(4)

,RMSE,MAE,n_avaliado
variavel,,,
DBO,10.5763,6.6941,14.3333
OD,3.9324,3.3608,12.3333
Nitrato,0.1668,0.0932,14.3333
Nitrogênio Amoniacal Total,2.8614,2.5798,13.3333
Fósforo Total,108.8725,34.0741,10.6667
Condutividade,17085.6089,13445.5707,13.6667
pH,0.6662,0.5260,17.6667
Turbidez,153.6535,56.5225,18.0000
Temperatura da Água,3.6782,3.1083,15.6667


## Baseline 4 — Forward fill por estação

Ordena as coletas de cada estação por data e **propaga o último valor observado** (`ffill`); resíduos no início da série recebem `bfill` (o próximo valor observado). O que sobra — variável sem nenhuma observação visível na estação dentro do test — cai para a média global do train.

Este é o único baseline simples que usa a **ordem temporal**. Se ganhar dos demais, a autocorrelação entre coletas consecutivas (~bimestrais) domina a estrutura do dado — e features como `dias_desde_inicio` merecem atenção na GAIN. A propagação opera apenas dentro do test (2024–2025), como no plano; não puxa histórico do train.

In [7]:
ordem_temporal = df_test.assign(_est=estacao_test.to_numpy()).sort_values(["_est", "Data"]).index

def predict_ffill_estacao(X_obs, seed=None):
    X_ord   = X_obs.loc[ordem_temporal]
    grupos  = estacao_test.loc[ordem_temporal].to_numpy()
    X_fill  = X_ord.groupby(grupos).ffill()
    X_fill  = X_fill.groupby(grupos).bfill()
    return X_fill.loc[X_obs.index].fillna(media_train)

met_ffill, pred = bu.avaliar_baseline(
    "ffill_estacao", predict_ffill_estacao, df_test, mask_real_test,
    VARS, scalers, transform_params, SEEDS, MISS_RATE, generate_artificial_mask)

pred.to_parquet(OUT_PRED_DIR / "predictions_ffill_estacao.parquet", index=False)
met_ffill.groupby("variavel", sort=False)[["RMSE", "MAE", "n_avaliado"]].mean().round(4)

,RMSE,MAE,n_avaliado
variavel,,,
DBO,6.1707,4.3500,14.3333
OD,3.6778,2.4199,12.3333
Nitrato,0.1432,0.0948,14.3333
Nitrogênio Amoniacal Total,4.0698,3.3415,13.3333
Fósforo Total,108.8634,33.9444,10.6667
Condutividade,14745.0671,9017.8072,13.6667
pH,0.8480,0.5831,17.6667
Turbidez,143.8179,48.0239,18.0000
Temperatura da Água,3.5513,2.5217,15.6667


## Consolidação

Concatena as métricas dos 4 métodos em `tabelas/baselines_simples.csv` (formato longo: `metodo, variavel, seed, miss_rate, n_avaliado, RMSE, MAE, RMSE_norm, MAE_norm`) e agrega média ± desvio entre sementes.

O RMSE em escala original **não é comparável entre variáveis** (mg/L de fósforo vs. µS/cm de condutividade); para o ranking global dos métodos usamos o `RMSE_norm` — erro no espaço normalizado `[-1, 1]`, onde todas as variáveis pesam igual.

In [8]:
met_simples = pd.concat(
    [met_media_global, met_media_estacao, met_mediana_global, met_ffill],
    ignore_index=True)
met_simples.to_csv(OUT_TAB_DIR / "baselines_simples.csv", index=False, encoding="utf-8")

n_esperado = 4 * 10 * len(SEEDS)  # 10 variáveis avaliáveis (Coliformes sem obs no test)
assert len(met_simples) == n_esperado, f"Esperava {n_esperado} linhas, obtive {len(met_simples)}."
print(f"baselines_simples.csv salvo — {len(met_simples)} linhas "
      f"(4 métodos × 10 variáveis avaliáveis × {len(SEEDS)} seeds).")

agg = bu.agregar_metricas(met_simples)
tabela_rmse = agg.pivot(index="variavel", columns="metodo", values="RMSE_media")
tabela_rmse["melhor_metodo"] = tabela_rmse.idxmin(axis=1)
tabela_rmse.round(3)

baselines_simples.csv salvo — 120 linhas (4 métodos × 10 variáveis avaliáveis × 3 seeds).


metodo,ffill_estacao,media_estacao,media_global,mediana_global,melhor_metodo
variavel,,,,,
Condutividade,14745.0670,10200.5090,15836.5420,17085.6090,media_estacao
DBO,6.1710,10.5860,10.7570,10.5760,ffill_estacao
Fósforo Total,108.8630,108.8580,108.9210,108.8720,media_estacao
Nitrato,0.1430,0.1640,0.1640,0.1670,ffill_estacao
Nitrogênio Amoniacal Total,4.0700,3.0490,2.7740,2.8610,media_global
OD,3.6780,4.0630,4.0230,3.9320,ffill_estacao
Sólidos Suspensos Totais,39.5830,31.9930,35.4280,33.5990,media_estacao
Temperatura da Água,3.5510,3.5090,3.4620,3.6780,media_global
Turbidez,143.8180,145.0200,153.4080,153.6530,ffill_estacao


In [9]:
# RMSE médio ± desvio por (método, variável) — visão completa do entregável inline
tabela_completa = agg.assign(
    RMSE=lambda d: d["RMSE_media"].round(3).astype(str) + " ± " + d["RMSE_std"].round(3).astype(str),
    MAE=lambda d: d["MAE_media"].round(3).astype(str) + " ± " + d["MAE_std"].round(3).astype(str),
).pivot(index="variavel", columns="metodo", values="RMSE")
tabela_completa

metodo,ffill_estacao,media_estacao,media_global,mediana_global
variavel,,,,
Condutividade,14745.067 ± 2396.527,10200.509 ± 3015.138,15836.542 ± 2067.686,17085.609 ± 1552.434
DBO,6.171 ± 1.375,10.586 ± 5.102,10.757 ± 5.31,10.576 ± 5.283
Fósforo Total,108.863 ± 187.974,108.858 ± 187.978,108.921 ± 187.832,108.872 ± 187.841
Nitrato,0.143 ± 0.085,0.164 ± 0.049,0.164 ± 0.05,0.167 ± 0.05
Nitrogênio Amoniacal Total,4.07 ± 0.104,3.049 ± 0.315,2.774 ± 0.117,2.861 ± 0.118
OD,3.678 ± 0.825,4.063 ± 0.336,4.023 ± 0.286,3.932 ± 0.292
Sólidos Suspensos Totais,39.583 ± 17.806,31.993 ± 8.459,35.428 ± 13.884,33.599 ± 11.875
Temperatura da Água,3.551 ± 0.937,3.509 ± 0.571,3.462 ± 0.564,3.678 ± 0.682
Turbidez,143.818 ± 107.505,145.02 ± 103.697,153.408 ± 103.458,153.653 ± 103.445


In [10]:
# Ranking global — RMSE_norm médio (espaço [-1, 1], comparável entre variáveis)
ranking = (met_simples.groupby("metodo")[["RMSE_norm", "MAE_norm"]]
           .mean().sort_values("RMSE_norm"))
ranking.round(4)

,RMSE_norm,MAE_norm
metodo,,
media_estacao,0.3497,0.2869
media_global,0.3762,0.3135
mediana_global,0.3850,0.3168
ffill_estacao,0.3979,0.2830


## Síntese final

### Ranking dos 4 baselines simples (RMSE_norm médio, espaço [-1, 1])

| # | Método | RMSE_norm | MAE_norm |
|---|--------|-----------|----------|
| 1 | **media_estacao** | **0,350** | 0,287 |
| 2 | media_global | 0,376 | 0,314 |
| 3 | mediana_global | 0,385 | 0,317 |
| 4 | ffill_estacao | 0,398 | **0,283** |

### Leitura

- **A média por estação é o baseline simples mais forte** — a referência mínima a ser batida pela GAIN (`RMSE_norm ≈ 0,350`). A vitória confirma quantitativamente a heterogeneidade espacial da EDA (§5): condicionar na estação reduz o RMSE_norm em ~7% sobre a média global. Recado para a Etapa 4: o one-hot `est_*` carrega sinal real.
- **O ffill conta uma história em duas métricas**: melhor MAE_norm (0,283) e pior RMSE_norm (0,398). O erro típico da propagação é pequeno (há autocorrelação entre coletas ~bimestrais), mas quando a propagação atravessa uma mudança de regime o erro é grande — e o RMSE, quadrático, pune exatamente isso. Por variável, o ffill vence em DBO, OD, Nitrato e Turbidez; a média por estação vence em pH, Condutividade, SST e Fósforo Total.
- **Fósforo Total é imune aos baselines**: RMSE ≈ 108,9 µg/L praticamente idêntico nos 4 métodos. O erro é dominado pela cauda superior (eventos de eutrofização) que nenhuma estatística central captura — exatamente o cenário do alerta duplo da Etapa 2. Este será o teste de fogo da GAIN em `04_GAIN/03_diagnostico.ipynb`.
- **Mediana ≈ média**: as transformações Box-Cox/Yeo-Johnson da Etapa 2 já simetrizaram as distribuições; a mediana não agrega nada no espaço normalizado.

### Critério de aceite

- ✅ 4 arquivos de predições em `Data/BaselineResults/`.
- ✅ `tabelas/baselines_simples.csv` com 120 linhas (4 métodos × 10 variáveis avaliáveis × 3 seeds — Coliformes sem observação no test, gap consciente).
- ✅ Baseline simples mais forte declarado: **média por estação**.

## Próximo notebook

`02_baselines_ml.ipynb` — KNN, MICE e MissForest: os concorrentes diretos da GAIN.